# Chapter 5: Queues and Deques

This notebook provides runnable JavaScript examples for queues, deques, circular queues, and practical interview-style problems.

## 1. Queue basics (FIFO)

Queues follow First In, First Out: new elements enter at the rear and leave from the front.

In [1]:
(() => {
  const queue = [];
  queue.push('a');
  queue.push('b');
  queue.push('c');

  console.log(queue.shift()); // a
  console.log(queue.shift()); // b
  console.log('remaining queue:', queue);
})();

a
b
remaining queue: [ 'c' ]


## 2. Queue class implementation

Array-based Queue with methods: enqueue, dequeue, front, isEmpty, size, clear, toString.

In [2]:
(() => {
  class Queue {
    #items = [];

    enqueue(item) {
      this.#items.push(item);
    }

    dequeue() {
      return this.#items.shift();
    }

    front() {
      return this.#items[0];
    }

    isEmpty() {
      return this.#items.length === 0;
    }

    get size() {
      return this.#items.length;
    }

    clear() {
      this.#items = [];
    }

    toString() {
      if (this.isEmpty()) return 'Empty Queue';
      return this.#items
        .map((item) => (typeof item === 'object' && item !== null ? JSON.stringify(item) : String(item)))
        .join(', ');
    }
  }

  const printQueue = new Queue();
  printQueue.enqueue({ document: 'Chapter05.docx', pages: 20 });
  printQueue.enqueue({ document: 'JavaScript.pdf', pages: 60 });
  printQueue.enqueue({ document: 'TypeScript.pdf', pages: 80 });

  console.log('front document:', printQueue.front());
  while (!printQueue.isEmpty()) {
    console.log('printing:', printQueue.dequeue());
  }
  console.log('after print:', printQueue.toString());
})();

front document: { document: 'Chapter05.docx', pages: 20 }
printing: { document: 'Chapter05.docx', pages: 20 }
printing: { document: 'JavaScript.pdf', pages: 60 }
printing: { document: 'TypeScript.pdf', pages: 80 }
after print: Empty Queue


## 3. Queue efficiency

- enqueue: O(1)
- dequeue (array shift): O(n)
- front, isEmpty, size, clear: O(1)
- toString: O(n)

## 4. Creating the Queue and Deque classes in TypeScript

With the JavaScript implementation done, we can rewrite our Queue and Deque classes using TypeScript:

```typescript
// src/05-queue-deque/queue.ts
class Queue<T> {
  private items: T[] = [];

  enqueue(item: T): void {}
  // all other methods are the same as in JavaScript
}

export default Queue;
```

And the Deque class:

```typescript
// src/05-queue-deque/deque.ts
class Deque<T> {
  private items: T[] = [];
  addFront(item: T): void {}
  addRear(item: T): void {}
  // all other methods are the same as in JavaScript
}

export default Deque;
```

We use generics to make these data structures flexible while still keeping all items in a queue or deque the same type. The method bodies are the same as the JavaScript implementation.

## 5. Deque basics

A deque (double-ended queue) allows insertion and removal at both front and rear.

In [3]:
(() => {
  class Deque {
    #items = [];

    addFront(item) {
      this.#items.unshift(item);
    }

    addRear(item) {
      this.#items.push(item);
    }

    removeFront() {
      return this.#items.shift();
    }

    removeRear() {
      return this.#items.pop();
    }

    peekFront() {
      return this.#items[0];
    }

    peekRear() {
      return this.#items[this.#items.length - 1];
    }

    isEmpty() {
      return this.#items.length === 0;
    }

    get size() {
      return this.#items.length;
    }

    clear() {
      this.#items = [];
    }

    toString() {
      if (this.isEmpty()) return 'Empty Deque';
      return this.#items.join(', ');
    }
  }

  const deque = new Deque();
  deque.addRear('B');
  deque.addFront('A');
  deque.addRear('C');
  console.log('peekFront:', deque.peekFront());
  console.log('peekRear:', deque.peekRear());
  console.log('removeFront:', deque.removeFront());
  console.log('removeRear:', deque.removeRear());
  console.log('deque now:', deque.toString());
})();

peekFront: A
peekRear: C
removeFront: A
removeRear: C
deque now: B


## 6. Browser history with deque

A deque can model navigation behavior when combined with current-page state.

In [4]:
(() => {
  class Deque {
    #items = [];
    addFront(item) { this.#items.unshift(item); }
    addRear(item) { this.#items.push(item); }
    removeFront() { return this.#items.shift(); }
    removeRear() { return this.#items.pop(); }
    peekFront() { return this.#items[0]; }
    peekRear() { return this.#items[this.#items.length - 1]; }
    get size() { return this.#items.length; }
  }

  class BrowserHistory {
    #history = new Deque();
    #currentPage = null;

    visit(url) {
      this.#history.addFront(url);
      this.#currentPage = url;
    }

    goBack() {
      if (this.#history.size > 1) {
        this.#history.removeFront();
        this.#currentPage = this.#history.peekFront();
      }
    }

    currentPage() {
      return this.#currentPage;
    }
  }

  const browser = new BrowserHistory();
  browser.visit('https://loiane.com');
  browser.visit('https://loiane.com/about');
  browser.goBack();
  console.log('current page:', browser.currentPage());
})();

current page: https://loiane.com


## 6. Circular Queue

Circular queues use fixed capacity and wrap pointers with modulo arithmetic, making enqueue/dequeue O(1).

In [5]:
(() => {
  class CircularQueue {
    #items;
    #capacity;
    #front = 0;
    #rear = -1;
    #size = 0;

    constructor(capacity) {
      this.#items = new Array(capacity);
      this.#capacity = capacity;
    }

    get size() {
      return this.#size;
    }

    isEmpty() {
      return this.#size === 0;
    }

    isFull() {
      return this.#size === this.#capacity;
    }

    enqueue(item) {
      if (this.isFull()) throw new Error('Queue is full');
      this.#rear = (this.#rear + 1) % this.#capacity;
      this.#items[this.#rear] = item;
      this.#size++;
    }

    dequeue() {
      if (this.isEmpty()) throw new Error('Queue is empty');
      const item = this.#items[this.#front];
      this.#size--;
      if (this.isEmpty()) {
        this.#front = 0;
        this.#rear = -1;
      } else {
        this.#front = (this.#front + 1) % this.#capacity;
      }
      return item;
    }
  }

  const cq = new CircularQueue(3);
  cq.enqueue('A');
  cq.enqueue('B');
  cq.enqueue('C');
  console.log(cq.dequeue());
  cq.enqueue('D');
  console.log(cq.dequeue());
  console.log(cq.dequeue());
  console.log(cq.dequeue());
})();

A
B
C
D


## 7. Hot Potato simulation

Players are passed around the queue and one is eliminated per round until one winner remains.

In [ ]:
(() => {
  class CircularQueue {
    #items; #capacity; #front = 0; #rear = -1; #size = 0;
    constructor(capacity) { this.#items = new Array(capacity); this.#capacity = capacity; }
    get size() { return this.#size; }
    isEmpty() { return this.#size === 0; }
    isFull() { return this.#size === this.#capacity; }
    enqueue(item) {
      if (this.isFull()) throw new Error('Queue is full');
      this.#rear = (this.#rear + 1) % this.#capacity;
      this.#items[this.#rear] = item;
      this.#size++;
    }
    dequeue() {
      if (this.isEmpty()) throw new Error('Queue is empty');
      const item = this.#items[this.#front];
      this.#size--;
      if (this.isEmpty()) { this.#front = 0; this.#rear = -1; }
      else { this.#front = (this.#front + 1) % this.#capacity; }
      return item;
    }
  }

  function hotPotato(players, numPasses) {
    const queue = new CircularQueue(players.length);
    for (const player of players) queue.enqueue(player);

    while (queue.size > 1) {
      for (let i = 0; i < numPasses; i++) {
        queue.enqueue(queue.dequeue());
      }
      console.log(`${queue.dequeue()} is eliminated!`);
    }
    return queue.dequeue();
  }

  const players = ['Violet', 'Feyre', 'Poppy', 'Oraya', 'Aelin'];
  const winner = hotPotato(players, 7);
  console.log(`The winner is: ${winner}!`);
})();

## 8. Palindrome checker with deque

Characters are compared from both ends: front and rear must match all the way through.

In [ ]:
(() => {
  class Deque {
    #items = [];
    addRear(item) { this.#items.push(item); }
    removeFront() { return this.#items.shift(); }
    removeRear() { return this.#items.pop(); }
    get size() { return this.#items.length; }
  }

  function isPalindrome(word) {
    if (word === undefined || word === null || (typeof word === 'string' && word.length === 0)) return false;

    const deque = new Deque();
    const normalized = word.toLowerCase().replace(//g, '');

    for (let i = 0; i < normalized.length; i++) deque.addRear(normalized[i]);

    while (deque.size > 1) {
      if (deque.removeFront() !== deque.removeRear()) return false;
    }
    return true;
  }

  console.log('racecar:', isPalindrome('racecar'));
  console.log('Never odd or even:', isPalindrome('Never odd or even'));
  console.log('JavaScript:', isPalindrome('JavaScript'));
})();

## 9. Exercise: Number of Students Unable to Eat Lunch

Queue simulation where students rotate until no one wants the current sandwich.

In [ ]:
(() => {
  function countStudents(students, sandwiches) {
    while (students.length > 0) {
      if (students[0] === sandwiches[0]) {
        students.shift();
        sandwiches.shift();
      } else {
        if (students.includes(sandwiches[0])) {
          const num = students.shift();
          students.push(num);
        } else {
          break;
        }
      }
    }
    return students.length;
  }

  console.log(countStudents([1, 1, 0, 0], [0, 1, 0, 1])); // 0
  console.log(countStudents([1, 1, 1, 0, 0, 1], [1, 0, 0, 0, 1, 1])); // 3
})();

## 10. Summary

You implemented queue, deque, and circular queue patterns, then applied them to realistic simulations and interview-style problems.